### Estimate Name-Conflict Overmerges

Estimates how many overmerged authors could be split based on incompatible name evidence alone.

**Approach**: Within a name block (e.g. "j priem"), if an author profile has works attributed to both "Jason Priem" and "Joseph Priem", it's provably overmerged. Uses `parsed_names_normalized` (#105.8) for robust name comparison including en-bloc middle initial matching.

**Two passes**:
1. Fast aggregate to flag authors with any name conflict
2. For flagged authors, identify core identity and outlier works, then filter out cross-indexing artifacts

**False positive filters** (discovered during initial run):
- Strip hyphens before first-name comparison ("yi-ting" = "yiting")
- Treat first names with length ≤ 2 as initials ("ml", "ak" are really M.L., A.K.)
- Exclude first names that match the author's most common last name (CJK surname/given-name parsing swaps)

Job: #105.2 aer-author-data-exploration

#### SQL UDF: `names_compatible`

Persistent UDF for comparing two parsed names. Returns TRUE if names could belong to the same person.

**Rules:**
- Last names must match exactly
- First names: strip hyphens first; NULL/empty → compatible; length ≤ 2 (initial) → compatible if same first char; both full → must be identical
- Middles: either has none (count=0) → compatible; both have middles → `middle_initials` strings must match **en bloc** ("rr" matches "robert randall" but NOT "r" or "rupert")

In [ ]:
%sql
CREATE OR REPLACE FUNCTION openalex.aer.names_compatible(
  first1 STRING, mid_initials1 STRING, mid_count1 INT, last1 STRING,
  first2 STRING, mid_initials2 STRING, mid_count2 INT, last2 STRING
)
RETURNS BOOLEAN
RETURN (
  -- Last names must match
  (last1 IS NOT NULL AND last2 IS NOT NULL AND last1 = last2)
  AND
  -- First names: strip hyphens, treat len ≤ 2 as initials
  CASE
    WHEN first1 IS NULL OR first2 IS NULL OR first1 = '' OR first2 = '' THEN TRUE
    WHEN LENGTH(REPLACE(first1, '-', '')) <= 2 OR LENGTH(REPLACE(first2, '-', '')) <= 2
      THEN SUBSTRING(REPLACE(first1, '-', ''), 1, 1) = SUBSTRING(REPLACE(first2, '-', ''), 1, 1)
    ELSE REPLACE(first1, '-', '') = REPLACE(first2, '-', '')
  END
  AND
  -- Middles: either has none → compatible; both present → initials must match en bloc
  CASE
    WHEN COALESCE(mid_count1, 0) = 0 OR COALESCE(mid_count2, 0) = 0 THEN TRUE
    ELSE COALESCE(mid_initials1, '') = COALESCE(mid_initials2, '')
  END
)

In [ ]:
%sql
-- Verify UDF with known test cases
SELECT
  first1, mid1, mc1, last1, first2, mid2, mc2, last2,
  openalex.aer.names_compatible(first1, mid1, mc1, last1, first2, mid2, mc2, last2) AS compatible,
  expected
FROM VALUES
  -- Same block, initial vs full → compatible
  ('j',       NULL, 0, 'priem', 'jason',   NULL, 0, 'priem', TRUE),
  -- Different full firsts → incompatible
  ('jason',   NULL, 0, 'priem', 'joseph',  NULL, 0, 'priem', FALSE),
  -- Full first + middle vs just full first → compatible (no middle info)
  ('jason',   'r',  1, 'priem', 'jason',   NULL, 0, 'priem', TRUE),
  -- Same first, different middles → incompatible
  ('jason',   'r',  1, 'priem', 'jason',   'm',  1, 'priem', FALSE),
  -- En bloc middle match: "rr" matches "rr"
  ('j',       'rr', 2, 'tolkien', 'john', 'rr', 2, 'tolkien', TRUE),
  -- En bloc middle mismatch: "rr" vs "r" (count differs)
  ('j',       'rr', 2, 'tolkien', 'j',    'r',  1, 'tolkien', FALSE),
  -- En bloc middle mismatch: "rr" vs "ra" (same count, different initials)
  ('j',       'rr', 2, 'tolkien', 'j',    'ra', 2, 'tolkien', FALSE),
  -- Different last names → incompatible (for cross-indexing check)
  ('jason',   NULL, 0, 'priem', 'jason',   NULL, 0, 'smith',  FALSE),
  -- Both initials only → compatible
  ('j',       NULL, 0, 'priem', 'j',       NULL, 0, 'priem',  TRUE),
  -- Initial + middle vs full + different middle → incompatible
  ('j',       'm',  1, 'priem', 'jason',   'r',  1, 'priem',  FALSE),
  -- Hyphenation variant: yi-ting vs yiting → compatible
  ('yi-ting',  NULL, 0, 'chen', 'yiting',   NULL, 0, 'chen', TRUE),
  -- 2-char name treated as initial: "li" compatible with "lingling"
  ('li',       NULL, 0, 'chen', 'lingling',  NULL, 0, 'chen', TRUE),
  -- 2-char initials: "ml" compatible with full name starting with same char
  ('ml',       NULL, 0, 'smith', 'michael',  NULL, 0, 'smith', TRUE)
AS t(first1, mid1, mc1, last1, first2, mid2, mc2, last2, expected)

#### Sample authors (works-weighted)

Sample ~100K rows from `work_authors` to get a works-weighted sample of authors. Prolific authors are more likely to be included (more rows), which is what we want.

In [ ]:
%sql
-- Sample ~100K work-author rows, then take distinct authors
-- RAND() < 0.0001 on 1.3B rows ≈ 130K rows → ~60-80K distinct authors
CREATE OR REPLACE TEMP VIEW sampled_authors AS
SELECT DISTINCT author_id
FROM openalex.works.work_authors
WHERE author_id IS NOT NULL AND RAND(42) < 0.0001;

SELECT COUNT(*) AS sampled_author_count FROM sampled_authors

#### Get all normalized names for sampled authors

For each sampled author, retrieve ALL their works (not just the sampled ones) and join to `parsed_names_normalized` for the normalized name fields.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW author_work_names AS
SELECT
  wa.author_id,
  wa.work_id,
  wa.raw_author_name,
  pn.normalized_first,
  pn.first_initial,
  pn.middle_initials,
  pn.middle_initial_count,
  pn.normalized_last
FROM openalex.works.work_authors wa
INNER JOIN sampled_authors sa ON wa.author_id = sa.author_id
INNER JOIN openalex.authors.parsed_names_normalized pn ON wa.raw_author_name = pn.raw_author_name;

SELECT
  COUNT(*) AS total_authorship_rows,
  COUNT(DISTINCT author_id) AS distinct_authors,
  COUNT(DISTINCT work_id) AS distinct_works
FROM author_work_names

#### Pass 1: Flag authors with name conflicts

Fast aggregate — no self-join. An author is flagged if they have:
- 2+ distinct full first names (length > 2 after hyphen removal, excluding surname-as-first), OR
- 2+ distinct middle initial patterns (where middle exists)

In [ ]:
%sql
-- First compute most common last name per author (for surname-swap filter)
CREATE OR REPLACE TEMP VIEW author_common_last AS
SELECT author_id, normalized_last AS common_last
FROM (
  SELECT author_id, normalized_last, COUNT(*) AS cnt,
    ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY COUNT(*) DESC) AS rn
  FROM author_work_names
  WHERE normalized_last IS NOT NULL
  GROUP BY author_id, normalized_last
) WHERE rn = 1;

CREATE OR REPLACE TEMP VIEW conflicted_authors AS
SELECT
  awn.author_id,
  COUNT(DISTINCT CASE
    WHEN LENGTH(REPLACE(awn.normalized_first, '-', '')) > 2
     AND REPLACE(awn.normalized_first, '-', '') != acl.common_last
    THEN REPLACE(awn.normalized_first, '-', '') END) AS distinct_full_firsts,
  COUNT(DISTINCT CASE WHEN awn.middle_initial_count > 0 THEN awn.middle_initials END) AS distinct_middles,
  COUNT(*) AS total_works,
  acl.common_last
FROM author_work_names awn
LEFT JOIN author_common_last acl ON awn.author_id = acl.author_id
GROUP BY awn.author_id, acl.common_last
HAVING
  COUNT(DISTINCT CASE
    WHEN LENGTH(REPLACE(awn.normalized_first, '-', '')) > 2
     AND REPLACE(awn.normalized_first, '-', '') != acl.common_last
    THEN REPLACE(awn.normalized_first, '-', '') END) > 1
  OR COUNT(DISTINCT CASE WHEN awn.middle_initial_count > 0 THEN awn.middle_initials END) > 1;

SELECT
  (SELECT COUNT(*) FROM sampled_authors) AS total_sampled,
  COUNT(*) AS conflicted_count,
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM sampled_authors), 1) AS conflicted_pct,
  SUM(CASE WHEN distinct_full_firsts > 1 AND distinct_middles <= 1 THEN 1 ELSE 0 END) AS first_only_conflicts,
  SUM(CASE WHEN distinct_full_firsts <= 1 AND distinct_middles > 1 THEN 1 ELSE 0 END) AS middle_only_conflicts,
  SUM(CASE WHEN distinct_full_firsts > 1 AND distinct_middles > 1 THEN 1 ELSE 0 END) AS both_conflicts
FROM conflicted_authors

#### Pass 2: Core identity + outlier works

For each conflicted author:
1. **Core first name** = most common full first name (length > 2 after hyphen removal)
2. **Core middle** = most common middle_initials (where count > 0)
3. **Core last name** = most common normalized_last

A work is an **outlier** if `names_compatible(work_name, core_name)` returns FALSE.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW outlier_works AS
WITH
core_first AS (
  SELECT author_id, clean_first AS core_first_name
  FROM (
    SELECT author_id, REPLACE(normalized_first, '-', '') AS clean_first, COUNT(*) AS cnt,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY COUNT(*) DESC) AS rn
    FROM author_work_names
    WHERE LENGTH(REPLACE(normalized_first, '-', '')) > 2
    GROUP BY author_id, REPLACE(normalized_first, '-', '')
  )
  WHERE rn = 1
),
core_middle AS (
  SELECT author_id, middle_initials AS core_mid_initials, middle_initial_count AS core_mid_count
  FROM (
    SELECT author_id, middle_initials, middle_initial_count, COUNT(*) AS cnt,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY COUNT(*) DESC) AS rn
    FROM author_work_names
    WHERE middle_initial_count > 0
    GROUP BY author_id, middle_initials, middle_initial_count
  )
  WHERE rn = 1
),
core_last AS (
  SELECT author_id, normalized_last AS core_last_name
  FROM (
    SELECT author_id, normalized_last, COUNT(*) AS cnt,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY COUNT(*) DESC) AS rn
    FROM author_work_names
    WHERE normalized_last IS NOT NULL
    GROUP BY author_id, normalized_last
  )
  WHERE rn = 1
),
core_identity AS (
  SELECT
    cl.author_id,
    cf.core_first_name,
    cm.core_mid_initials,
    COALESCE(cm.core_mid_count, 0) AS core_mid_count,
    cl.core_last_name
  FROM core_last cl
  INNER JOIN conflicted_authors ca ON cl.author_id = ca.author_id
  LEFT JOIN core_first cf ON cl.author_id = cf.author_id
  LEFT JOIN core_middle cm ON cl.author_id = cm.author_id
)
SELECT
  awn.author_id,
  awn.work_id,
  awn.raw_author_name,
  awn.normalized_first,
  awn.middle_initials,
  awn.middle_initial_count,
  awn.normalized_last,
  ci.core_first_name,
  ci.core_mid_initials,
  ci.core_mid_count,
  ci.core_last_name,
  CASE
    WHEN (LENGTH(REPLACE(awn.normalized_first, '-', '')) > 2 AND ci.core_first_name IS NOT NULL
          AND REPLACE(awn.normalized_first, '-', '') != ci.core_first_name
          AND REPLACE(awn.normalized_first, '-', '') != ci.core_last_name)
     AND (awn.middle_initial_count > 0 AND ci.core_mid_initials IS NOT NULL AND awn.middle_initials != ci.core_mid_initials)
      THEN 'both'
    WHEN LENGTH(REPLACE(awn.normalized_first, '-', '')) > 2 AND ci.core_first_name IS NOT NULL
         AND REPLACE(awn.normalized_first, '-', '') != ci.core_first_name
         AND REPLACE(awn.normalized_first, '-', '') != ci.core_last_name
      THEN 'first_name'
    WHEN awn.middle_initial_count > 0 AND ci.core_mid_initials IS NOT NULL AND awn.middle_initials != ci.core_mid_initials
      THEN 'middle'
    WHEN awn.normalized_last != ci.core_last_name THEN 'last_name'
    ELSE 'initial_mismatch'
  END AS conflict_type
FROM author_work_names awn
INNER JOIN core_identity ci ON awn.author_id = ci.author_id
WHERE NOT openalex.aer.names_compatible(
  awn.normalized_first, awn.middle_initials, awn.middle_initial_count, awn.normalized_last,
  ci.core_first_name, ci.core_mid_initials, ci.core_mid_count, ci.core_last_name
);

SELECT
  COUNT(*) AS total_outlier_works,
  COUNT(DISTINCT author_id) AS authors_with_outliers,
  SUM(CASE WHEN conflict_type = 'first_name' THEN 1 ELSE 0 END) AS first_name,
  SUM(CASE WHEN conflict_type = 'middle' THEN 1 ELSE 0 END) AS middle,
  SUM(CASE WHEN conflict_type = 'both' THEN 1 ELSE 0 END) AS both,
  SUM(CASE WHEN conflict_type = 'last_name' THEN 1 ELSE 0 END) AS last_name,
  SUM(CASE WHEN conflict_type = 'initial_mismatch' THEN 1 ELSE 0 END) AS initial_mismatch
FROM outlier_works

#### Cross-indexing filter

~5% of authors got cross-indexed with coauthors (name attached to wrong position). For each outlier work, check if ANY coauthor on that paper has a name compatible with the author's core identity. If yes → likely a position swap, not a real overmerge.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW cross_index_matches AS
SELECT DISTINCT ow.author_id, ow.work_id
FROM outlier_works ow
INNER JOIN openalex.works.work_authors coauthor
  ON ow.work_id = coauthor.work_id
  AND coauthor.author_id != ow.author_id
INNER JOIN openalex.authors.parsed_names_normalized cpn
  ON coauthor.raw_author_name = cpn.raw_author_name
WHERE openalex.aer.names_compatible(
  cpn.normalized_first, cpn.middle_initials, cpn.middle_initial_count, cpn.normalized_last,
  ow.core_first_name, ow.core_mid_initials, ow.core_mid_count, ow.core_last_name
);

CREATE OR REPLACE TEMP VIEW genuine_outlier_works AS
SELECT ow.*
FROM outlier_works ow
LEFT JOIN cross_index_matches cim
  ON ow.author_id = cim.author_id AND ow.work_id = cim.work_id
WHERE cim.work_id IS NULL;

SELECT
  (SELECT COUNT(*) FROM outlier_works) AS pre_filter_outlier_works,
  (SELECT COUNT(DISTINCT author_id) FROM outlier_works) AS pre_filter_authors,
  (SELECT COUNT(*) FROM cross_index_matches) AS cross_index_excluded,
  COUNT(*) AS post_filter_outlier_works,
  COUNT(DISTINCT author_id) AS post_filter_authors
FROM genuine_outlier_works

#### Metrics

In [ ]:
%sql
-- Overmerge rate by works_count bucket
-- This is the key output: how overmerge rate scales with profile size
WITH author_stats AS (
  SELECT
    sa.author_id,
    COALESCE(awn_counts.total_works, 0) AS total_works,
    CASE WHEN go.author_id IS NOT NULL THEN 1 ELSE 0 END AS is_overmerged
  FROM sampled_authors sa
  LEFT JOIN (
    SELECT author_id, COUNT(*) AS total_works
    FROM author_work_names GROUP BY author_id
  ) awn_counts ON sa.author_id = awn_counts.author_id
  LEFT JOIN (
    SELECT DISTINCT author_id FROM genuine_outlier_works
  ) go ON sa.author_id = go.author_id
)
SELECT
  CASE
    WHEN total_works <= 1 THEN '1'
    WHEN total_works <= 5 THEN '2-5'
    WHEN total_works <= 20 THEN '6-20'
    WHEN total_works <= 50 THEN '21-50'
    WHEN total_works <= 200 THEN '51-200'
    ELSE '200+'
  END AS works_bucket,
  COUNT(*) AS sampled_authors,
  SUM(is_overmerged) AS overmerged,
  ROUND(100.0 * SUM(is_overmerged) / COUNT(*)) AS overmerge_pct
FROM author_stats
GROUP BY 1
ORDER BY MIN(total_works)

In [ ]:
%sql
-- Work-level summary
SELECT
  COUNT(*) AS genuine_outlier_works,
  COUNT(DISTINCT author_id) AS affected_authors,
  SUM(CASE WHEN conflict_type = 'first_name' THEN 1 ELSE 0 END) AS first_name,
  SUM(CASE WHEN conflict_type = 'middle' THEN 1 ELSE 0 END) AS middle,
  SUM(CASE WHEN conflict_type = 'both' THEN 1 ELSE 0 END) AS both,
  SUM(CASE WHEN conflict_type = 'last_name' THEN 1 ELSE 0 END) AS last_name,
  SUM(CASE WHEN conflict_type = 'initial_mismatch' THEN 1 ELSE 0 END) AS initial_mismatch
FROM genuine_outlier_works

In [ ]:
%sql
-- Severity distribution: outlier works per affected author
SELECT
  outlier_bucket, num_authors,
  ROUND(100.0 * num_authors / SUM(num_authors) OVER ()) AS pct
FROM (
  SELECT
    CASE
      WHEN cnt = 1 THEN '01: 1 work'
      WHEN cnt <= 5 THEN '02: 2-5 works'
      WHEN cnt <= 20 THEN '03: 6-20 works'
      WHEN cnt <= 100 THEN '04: 21-100 works'
      ELSE '05: 100+ works'
    END AS outlier_bucket,
    COUNT(*) AS num_authors
  FROM (
    SELECT author_id, COUNT(*) AS cnt
    FROM genuine_outlier_works
    GROUP BY author_id
  )
  GROUP BY 1
)
ORDER BY outlier_bucket

#### Spot-check: inspect flagged authors

Manual verification — look at actual name variants for a sample of flagged authors to confirm they're real overmerges.

In [ ]:
%sql
-- Show 20 flagged authors with their core identity and outlier names
SELECT
  gow.author_id,
  gow.core_first_name,
  gow.core_mid_initials,
  gow.core_last_name,
  gow.conflict_type,
  gow.normalized_first AS outlier_first,
  gow.middle_initials AS outlier_mid,
  gow.raw_author_name AS outlier_raw_name,
  ca.total_works
FROM genuine_outlier_works gow
INNER JOIN conflicted_authors ca ON gow.author_id = ca.author_id
WHERE ca.total_works BETWEEN 5 AND 200
ORDER BY RAND(77)
LIMIT 20